# Lab | Langchain Evaluation

## Intro

Pick different sets of data and re-run this notebook. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications.

What did you learn? - Let's discuss that in class

## LangChain: Evaluation

### Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation

In [2]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY') 

### Example 1

#### Create our QandA application

In [3]:
pip install langchain-huggingface

In [4]:
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from langchain.llms import OpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.document_loaders import CSVLoader, TextLoader
from langchain.indexes import VectorstoreIndexCreator
from langchain.vectorstores import DocArrayInMemorySearch
from langchain.chains import LLMChain


In [5]:
import os
print(os.getcwd())

c:\Users\User\Desktop\AI_Lab\WK8-Labs\lab-langchain-evaluation


In [6]:
import os
print(os.listdir('.'))

['.env', '.git', 'data', 'lab-langchain-evaluation.ipynb', 'README.md']


In [7]:
file = 'data/OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file)
data = loader.load()

In [8]:
# !pip install --upgrade --force-reinstall sentence-transformers

In [9]:
pip install docarray

In [10]:
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs = {'device': 'cpu'})
).from_loaders([loader])

c:\Users\User\anaconda3\Lib\site-packages\pydantic\_migration.py:283: UserWarning: `pydantic.error_wrappers:ValidationError` has been moved to `pydantic:ValidationError`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


In [11]:
llm = ChatOpenAI(temperature = 0.0)
qa = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=index.vectorstore.as_retriever(), 
    verbose=True,
    chain_type_kwargs = {
        "document_separator": "<<<<>>>>>"
    }
)

#### Coming up with test datapoints

In [12]:
data[10]

Document(metadata={'source': 'data/OutdoorClothingCatalog_1000.csv', 'row': 10}, page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\r\n\r\nSize & Fit\r\n- Pants are Favorite Fit: Sits lower on the waist.\r\n- Relaxed Fit: Our most generous fit sits farthest from the body.\r\n\r\nFabric & Care\r\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\r\n\r\nAdditional Features\r\n- Relaxed fit top with raglan sleeves and rounded hem.\r\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\r\n\r\nImported.")

In [13]:
data[11]

Document(metadata={'source': 'data/OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.')

#### Hard-coded examples

In [14]:
from langchain.prompts import PromptTemplate

In [15]:
from langchain.prompts import PromptTemplate
from langchain.schema import BaseOutputParser
from pydantic import BaseModel, Field

examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set\
        have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty \
        850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

# Define the prompt template
prompt_template = PromptTemplate(
    input_variables=["query"],
    template="Examples:\n"
             "1. Query: Do the Cozy Comfort Pullover Set have side pockets?\n"
             "   Answer: Yes\n"
             "2. Query: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?\n"
             "   Answer: The DownTek collection\n"
             "Query: {query}\n"
             "Answer:"
)

# Define the output model
class Answer(BaseModel):
    answer: str = Field(description="The answer to the query")

# Create the output parser
class AnswerOutputParser(BaseOutputParser):
    def parse(self, text: str) -> Answer:
        # Split the response to get the answer
        answer = text.strip().split("Answer:")[-1].strip()
        return Answer(answer=answer)

# Initialize the LLM
# llm = OpenAI()
llm = ChatOpenAI()

# Create the LLMChain
llm_chain = LLMChain(
    llm=llm,
    prompt=prompt_template,
    output_parser=AnswerOutputParser()
)

# Example query
query = "Is the Cozy Comfort Pullover Set available in different colors?"

# Run the chain
result = llm_chain.run({"query": query})

# Print the result
print(result)


C:\Users\User\AppData\Local\Temp\ipykernel_5252\1856079041.py:46: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(
C:\Users\User\AppData\Local\Temp\ipykernel_5252\1856079041.py:56: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = llm_chain.run({"query": query})


answer='Yes, it is available in black, gray, and navy.'


#### LLM-Generated examples

In [16]:
from langchain.evaluation.qa import QAGenerateChain

In [17]:
example_gen_chain = QAGenerateChain.from_llm(ChatOpenAI())

In [18]:
llm_chain = LLMChain(llm=llm, prompt=prompt_template)

In [19]:
new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

C:\Users\User\AppData\Local\Temp\ipykernel_5252\3125874183.py:1: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  new_examples = example_gen_chain.apply_and_parse(


In [20]:
new_examples[0]

{'qa_pairs': {'query': "What is the approximate weight of the Women's Campside Oxfords per pair?",
  'answer': "The approximate weight of the Women's Campside Oxfords is 1 lb. 1 oz. per pair."}}

In [21]:
data[0]

Document(metadata={'source': 'data/OutdoorClothingCatalog_1000.csv', 'row': 0}, page_content=": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \r\n\r\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \r\n\r\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \r\n\r\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. \r\n\r\nQuestions? Please contact us for any inquiries.")

In [22]:
d_flattened = [data['qa_pairs'] for data in new_examples]
d_flattened

[{'query': "What is the approximate weight of the Women's Campside Oxfords per pair?",
  'answer': "The approximate weight of the Women's Campside Oxfords is 1 lb. 1 oz. per pair."},
 {'query': 'What are the dimensions of the Small and Medium sizes for the Recycled Waterhog Dog Mat, Chevron Weave?',
  'answer': 'The Small size has dimensions of 18" x 28" and the Medium size has dimensions of 22.5" x 34.5".'},
 {'query': "What features are included in the Infant and Toddler Girls' Coastal Chill Swimsuit, Two-Piece as described in the document?",
  'answer': "The features included in the Infant and Toddler Girls' Coastal Chill Swimsuit, Two-Piece are bright colors, ruffles, exclusive whimsical prints, four-way-stretch and chlorine-resistant fabric, UPF 50+ rated fabric for sun protection, crossover no-slip straps, fully lined bottom for secure fit and maximum coverage. It is recommended to machine wash and line dry for best results."},
 {'query': 'What is the fabric composition of the Re

#### Combine examples

In [23]:
# examples += new_example
examples += d_flattened

In [24]:
examples[0]

{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
 'answer': 'Yes'}

In [25]:
qa.invoke(examples[0]["query"])



> Entering new RetrievalQA chain...



> Finished chain.


{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
 'result': 'Based on the provided context, the Cozy Comfort Pullover Set does not have side pockets. It has side seam pockets, a back zip pocket, two elastic mesh water bottle pockets, and a top compartment with a pocket with a double-seal zipper for quick access.'}

### Manual Evaluation - Fun part

In [26]:
import langchain
langchain.debug = True

In [27]:
qa.invoke(examples[0]["query"])

[chain/start] [chain:RetrievalQA] Entering Chain run with input:
{
  "query": "Do the Cozy Comfort Pullover Set        have side pockets?"
}
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain] Entering Chain run with input:
[inputs]
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain] Entering Chain run with input:
{
  "question": "Do the Cozy Comfort Pullover Set        have side pockets?",
  "context": "Side seam pockets and back zip pocket, with mesh insert for quick drainage.<<<<>>>>>Two elastic mesh water bottle pockets.\r\nTop compartment includes pocket with double-seal zipper for quick access.\r\nSide<<<<>>>>>All pockets have sturdy pocket bags and offer plenty of room for a wallet, cell phone and more.\r\n\r\nGusseted crotch for ease of movement.\r\n\r\nImported.<<<<>>>>>Two elastic mesh water bottle pockets.\r\nTop compartment includes pocket with double-se"
}
[llm/start] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain > llm

[llm/end] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain > llm:ChatOpenAI] s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "Based on the provided context, the Cozy Comfort Pullover Set does not have side pockets. It has side seam pockets, a back zip pocket, two elastic mesh water bottle pockets, and a top compartment with a pocket with a double-seal zipper for quick access.",
        "generation_info": {
          "finish_reason": "stop",
          "logprobs": null
        },
        "type": "ChatGeneration",
        "message": {
          "lc": 1,
          "type": "constructor",
          "id": [
            "langchain",
            "schema",
            "messages",
            "AIMessage"
          ],
          "kwargs": {
            "content": "Based on the provided context, the Cozy Comfort Pullover Set does not have side pockets. It has side seam pockets, a back zip pocket, two elastic mesh water bottle pockets, and a top compa

{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
 'result': 'Based on the provided context, the Cozy Comfort Pullover Set does not have side pockets. It has side seam pockets, a back zip pocket, two elastic mesh water bottle pockets, and a top compartment with a pocket with a double-seal zipper for quick access.'}

In [28]:
# Turn off the debug mode
langchain.debug = False

### LLM assisted evaluation

In [29]:
examples += d_flattened

In [30]:
examples

[{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
  'answer': 'Yes'},
 {'query': 'What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?',
  'answer': 'The DownTek collection'},
 {'query': "What is the approximate weight of the Women's Campside Oxfords per pair?",
  'answer': "The approximate weight of the Women's Campside Oxfords is 1 lb. 1 oz. per pair."},
 {'query': 'What are the dimensions of the Small and Medium sizes for the Recycled Waterhog Dog Mat, Chevron Weave?',
  'answer': 'The Small size has dimensions of 18" x 28" and the Medium size has dimensions of 22.5" x 34.5".'},
 {'query': "What features are included in the Infant and Toddler Girls' Coastal Chill Swimsuit, Two-Piece as described in the document?",
  'answer': "The features included in the Infant and Toddler Girls' Coastal Chill Swimsuit, Two-Piece are bright colors, ruffles, exclusive whimsical prints, four-way-stretch and chlorine-resistant fabric, UPF 50+ rated fab

In [31]:
predictions = qa.batch(examples)



> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...



> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.


In [32]:
predictions

[{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
  'answer': 'Yes',
  'result': 'Based on the provided context, the Cozy Comfort Pullover Set does not have side pockets. It has side seam pockets and a back zip pocket, but not side pockets.'},
 {'query': 'What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?',
  'answer': 'The DownTek collection',
  'result': 'The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.'},
 {'query': "What is the approximate weight of the Women's Campside Oxfords per pair?",
  'answer': "The approximate weight of the Women's Campside Oxfords is 1 lb. 1 oz. per pair.",
  'result': "The approximate weight of the Women's Campside Oxfords per pair is 1 lb. 1 oz."},
 {'query': 'What are the dimensions of the Small and Medium sizes for the Recycled Waterhog Dog Mat, Chevron Weave?',
  'answer': 'The Small size has dimensions of 18" x 28" and the Medium size has dimensions of 22.5" x 34.5".',


In [33]:
from langchain.evaluation.qa import QAEvalChain

In [34]:
llm = ChatOpenAI(temperature=0)
eval_chain = QAEvalChain.from_llm(llm)

In [35]:
graded_outputs = eval_chain.evaluate(examples, predictions)

In [36]:
graded_outputs

[{'results': 'INCORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'}]

In [37]:
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print("Question: " + predictions[i]['query'])
    print("Real Answer: " + predictions[i]['answer'])
    print("Predicted Answer: " + predictions[i]['result'])
    # print("Predicted Grade: " + graded_outputs[i]['text'])
    print()

Example 0:
Question: Do the Cozy Comfort Pullover Set        have side pockets?
Real Answer: Yes
Predicted Answer: Based on the provided context, the Cozy Comfort Pullover Set does not have side pockets. It has side seam pockets and a back zip pocket, but not side pockets.

Example 1:
Question: What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?
Real Answer: The DownTek collection
Predicted Answer: The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.

Example 2:
Question: What is the approximate weight of the Women's Campside Oxfords per pair?
Real Answer: The approximate weight of the Women's Campside Oxfords is 1 lb. 1 oz. per pair.
Predicted Answer: The approximate weight of the Women's Campside Oxfords per pair is 1 lb. 1 oz.

Example 3:
Question: What are the dimensions of the Small and Medium sizes for the Recycled Waterhog Dog Mat, Chevron Weave?
Real Answer: The Small size has dimensions of 18" x 28" and the Medium size has 

### Example 2
One can also easily evaluate your QA chains with the metrics offered in ragas

In [38]:
pip install numpy==1.26.4

Note: you may need to restart the kernel to use updated packages.


In [39]:
import os
print(os.listdir("data"))

['nyc_text.txt', 'OutdoorClothingCatalog_1000.csv']


In [40]:
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from langchain.llms import OpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.document_loaders import CSVLoader, TextLoader
from langchain.indexes import VectorstoreIndexCreator
from langchain.vectorstores import DocArrayInMemorySearch

loader = TextLoader("data/nyc_text.txt")
index = VectorstoreIndexCreator(
    embedding=HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2",
        model_kwargs = {'device': 'cpu'} # Change 'mps' to 'cpu'
)
).from_loaders([loader])

llm = ChatOpenAI(temperature=0)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=index.vectorstore.as_retriever(),
return_source_documents=True,
)

C:\Users\User\AppData\Local\Temp\ipykernel_5252\2483884132.py:10: UserWarning: Using InMemoryVectorStore as the default vectorstore.This memory store won't persist data. You should explicitlyspecify a vectorstore when using VectorstoreIndexCreator
  index = VectorstoreIndexCreator(


In [41]:
# testing it out

question = "How did New York City get its name?"
result = qa_chain.invoke({"query": question})
result["result"]

'New York City was originally named New Amsterdam by Dutch colonists in 1626. When the city came under British control in 1664, it was renamed New York after King Charles II of England granted the lands to his brother, the Duke of York. The city has been continuously named New York since November 1674.'

In [42]:
result

{'query': 'How did New York City get its name?',
 'result': 'New York City was originally named New Amsterdam by Dutch colonists in 1626. When the city came under British control in 1664, it was renamed New York after King Charles II of England granted the lands to his brother, the Duke of York. The city has been continuously named New York since November 1674.',
 'source_documents': [Document(id='6e0f71a4-08c3-4145-9218-1b0573287b73', metadata={'source': 'data/nyc_text.txt'}, page_content='The city and its metropolitan area constitute the premier gateway for legal immigration to the United States. As many as 800 languages are spoken in New York, making it the most linguistically diverse city in the world. New York City is home to more than 3.2 million residents born outside the U.S., the largest foreign-born population of any city in the world as of 2016.New York City traces its origins to a trading post founded on the southern tip of Manhattan Island by Dutch colonists in approximate

Now in order to evaluate the qa system we generated a few relevant questions. We've generated a few question for you but feel free to add any you want.

In [43]:
eval_questions = [
    "What is the population of New York City as of 2020?",
    "Which borough of New York City has the highest population?",
    "What is the economic significance of New York City?",
    "How did New York City get its name?",
    "What is the significance of the Statue of Liberty in New York City?",
]

eval_answers = [
    "8,804,190",
    "Brooklyn",
    "New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter.",
    "New York City got its name when it came under British control in 1664. King Charles II of England granted the lands to his brother, the Duke of York, who named the city New York in his own honor.",
    "The Statue of Liberty in New York City holds great significance as a symbol of the United States and its ideals of liberty and peace. It greeted millions of immigrants who arrived in the U.S. by ship in the late 19th and early 20th centuries, representing hope and freedom for those seeking a better life. It has since become an iconic landmark and a global symbol of cultural diversity and freedom.",
]

examples = [
    {"query": q, "ground_truths": [eval_answers[i]]}
    for i, q in enumerate(eval_questions)
]

In [44]:
examples

[{'query': 'What is the population of New York City as of 2020?',
  'ground_truths': ['8,804,190']},
 {'query': 'Which borough of New York City has the highest population?',
  'ground_truths': ['Brooklyn']},
 {'query': 'What is the economic significance of New York City?',
  'ground_truths': ["New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter."]},
 {'query': 'How did New York City

#### Introducing RagasEvaluatorChain

`RagasEvaluatorChain` creates a wrapper around the metrics ragas provides (documented [here](https://github.com/explodinggradients/ragas/blob/main/docs/metrics.md)), making it easier to run these evaluation with langchain and langsmith.

The evaluator chain has the following APIs

- `__call__()`: call the `RagasEvaluatorChain` directly on the result of a QA chain.
- `evaluate()`: evaluate on a list of examples (with the input queries) and predictions (outputs from the QA chain). 
- `evaluate_run()`: method implemented that is called by langsmith evaluators to evaluate langsmith datasets.

lets see each of them in action to learn more.

In [45]:
result = qa.invoke({"query": eval_questions[1]})
result["result"]



> Entering new RetrievalQA chain...



> Finished chain.


"I don't know."

In [46]:
key_mapping = {
    "query": "question",
    "result": "answer",
    "source_documents": "contexts"
}

result_updated = {}
for old_key, new_key in key_mapping.items():
    if old_key in result:
        result_updated[new_key] = result[old_key]


In [47]:
result_updated

{'question': 'Which borough of New York City has the highest population?',
 'answer': "I don't know."}

In [48]:
# !pip install --no-cache-dir recordclass

In [49]:
# !pip install ragas==0.1.9

In [50]:
pip install --upgrade --force-reinstall ragas


  Using cached ragas-0.3.3-py3-none-any.whl.metadata (21 kB)
  Using cached numpy-2.3.2-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached datasets-4.0.0-py3-none-any.whl.metadata (19 kB)
  Using cached tiktoken-0.11.0-cp312-cp312-win_amd64.whl.metadata (6.9 kB)
  Using cached pydantic-2.11.7-py3-none-any.whl.metadata (67 kB)
  Using cached nest_asyncio-1.6.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
  Using cached typer-0.17.4-py3-none-any.whl.metadata (15 kB)
  Using cached rich-14.1.0-py3-none-any.whl.metadata (18 kB)
  Using cached openai-1.106.1-py3-none-any.whl.metadata (29 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached instructor-1.11.2-py3-none-any.whl.metadata (11 kB)
  Using cached gitpython-3.1.45-py3-none-any.whl.metadata (13 kB)
  Using cached pillow-11.3.0-cp312-cp312-win_amd64.whl.metadata (9.2 kB)
  Using 

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
E

In [51]:
pip install ragas

In [52]:
pip show ragas

Name: ragas
Version: 0.3.3
Summary: Evaluation framework for RAG and LLM applications
Home-page: https://github.com/explodinggradients/ragas
Author: 
Author-email: 
License: Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized byNote: you may need to restart the kernel to use updated packages.

      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect

In [53]:
pip install numpy==1.26.4

Note: you may need to restart the kernel to use updated packages.Collecting numpy==1.26.4
  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl (15.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.2
    Uninstalling numpy-2.3.2:
      Successfully uninstalled numpy-2.3.2



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.16.1 which is incompatible.
streamlit 1.37.1 requires packaging<25,>=20, but you have packaging 25.0 which is incompatible.
streamlit 1.37.1 requires pillow<11,>=7.1.0, but you have pillow 11.3.0 which is incompatible.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 6.32.0 which is incompatible.
streamlit 1.37.1 requires rich<14,>=10.14.0, but you have rich 14.1.0 which is incompatible.
streamlit 1.37.1 requires tenacity<9,>=8.1.0, but you have tenacity 9.1.2 which is incompatible.


In [54]:
import sys
import shutil
print(f"Python executable: {sys.executable}")
print(f"Pip executable: {shutil.which('pip')}")

Python executable: c:\Users\User\anaconda3\python.exe
Pip executable: C:\Users\User\anaconda3\Scripts\pip.EXE


In [55]:
!python -m pip install --upgrade --force-reinstall ragas

  Using cached ragas-0.3.3-py3-none-any.whl.metadata (21 kB)
  Using cached numpy-2.3.2-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached datasets-4.0.0-py3-none-any.whl.metadata (19 kB)
  Using cached tiktoken-0.11.0-cp312-cp312-win_amd64.whl.metadata (6.9 kB)
  Using cached pydantic-2.11.7-py3-none-any.whl.metadata (67 kB)
  Using cached nest_asyncio-1.6.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
  Using cached typer-0.17.4-py3-none-any.whl.metadata (15 kB)
  Using cached rich-14.1.0-py3-none-any.whl.metadata (18 kB)
  Using cached openai-1.106.1-py3-none-any.whl.metadata (29 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached instructor-1.11.2-py3-none-any.whl.metadata (11 kB)
  Using cached gitpython-3.1.45-py3-none-any.whl.metadata (13 kB)
  Using cached pillow-11.3.0-cp312-cp312-win_amd64.whl.metadata (9.2 kB)
  Using c

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.3.2 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.3.2 which is incompatible.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.16.1 which is incompatible.
mdit-py-plugins 0.3.0 requires markdown-it-py<3.0.0,>=1.0.0, but you have markdown-it-py 4.0.0 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.2 which is incompatible.
s3fs 2024.6.1 requires fsspec==2024.6.1.*, but you have fsspec 2025.3.0 which is incompatible.
streamlit 1.37.1 requires packaging<25,>=20, but you have packaging 25.0 which is incompatible.
streamlit 1.37.1 requires pillow<11,>=7.1.0, but you have pillow 11.3.0 which is incompatible.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have p

In [56]:
!python -m pip install --upgrade --force-reinstall ragas
!python -m pip install --upgrade --force-reinstall "numpy<2" sentence-transformers docarray

  Using cached ragas-0.3.3-py3-none-any.whl.metadata (21 kB)
  Using cached numpy-2.3.2-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached datasets-4.0.0-py3-none-any.whl.metadata (19 kB)
  Using cached tiktoken-0.11.0-cp312-cp312-win_amd64.whl.metadata (6.9 kB)
  Using cached pydantic-2.11.7-py3-none-any.whl.metadata (67 kB)
  Using cached nest_asyncio-1.6.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
  Using cached typer-0.17.4-py3-none-any.whl.metadata (15 kB)
  Using cached rich-14.1.0-py3-none-any.whl.metadata (18 kB)
  Using cached openai-1.106.1-py3-none-any.whl.metadata (29 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached instructor-1.11.2-py3-none-any.whl.metadata (11 kB)
  Using cached gitpython-3.1.45-py3-none-any.whl.metadata (13 kB)
  Using cached pillow-11.3.0-cp312-cp312-win_amd64.whl.metadata (9.2 kB)
  Using c

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.3.2 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.3.2 which is incompatible.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.16.1 which is incompatible.
mdit-py-plugins 0.3.0 requires markdown-it-py<3.0.0,>=1.0.0, but you have markdown-it-py 4.0.0 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.2 which is incompatible.
s3fs 2024.6.1 requires fsspec==2024.6.1.*, but you have fsspec 2025.3.0 which is incompatible.
streamlit 1.37.1 requires packaging<25,>=20, but you have packaging 25.0 which is incompatible.
streamlit 1.37.1 requires pillow<11,>=7.1.0, but you have pillow 11.3.0 which is incompatible.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have p

  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
  Using cached sentence_transformers-5.1.0-py3-none-any.whl.metadata (16 kB)
  Using cached docarray-0.41.0-py3-none-any.whl.metadata (36 kB)
  Using cached transformers-4.56.1-py3-none-any.whl.metadata (42 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached torch-2.8.0-cp312-cp312-win_amd64.whl.metadata (30 kB)
  Using cached scikit_learn-1.7.1-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.16.1-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached huggingface_hub-0.34.4-py3-none-any.whl.metadata (14 kB)
  Using cached pillow-11.3.0-cp312-cp312-win_amd64.whl.metadata (9.2 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached filelock-3.19.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached packaging-25.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached PyYAML-6.0.2-cp312-cp312-win_amd64.whl.metadata (2.1 kB)
  Using cached r

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2025.9.0 which is incompatible.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.16.1 which is incompatible.
mdit-py-plugins 0.3.0 requires markdown-it-py<3.0.0,>=1.0.0, but you have markdown-it-py 4.0.0 which is incompatible.
s3fs 2024.6.1 requires fsspec==2024.6.1.*, but you have fsspec 2025.9.0 which is incompatible.
streamlit 1.37.1 requires packaging<25,>=20, but you have packaging 25.0 which is incompatible.
streamlit 1.37.1 requires pillow<11,>=7.1.0, but you have pillow 11.3.0 which is incompati

In [59]:
# RAGAS x LangChain evaluator chains (works across versions)

# 1) Import the evaluator wrapper with a robust fallback
try:
    # Newer/most-current docs often use this path
    from ragas.langchain.evalchain import RagasEvaluatorChain as EvaluatorChain
except Exception:
    # Older docs / some builds expose it here
    from ragas.integrations.langchain import EvaluatorChain

# 2) Import metrics (note: context_relevancy → context_precision)
try:
    from ragas.metrics import (
        faithfulness,
        answer_relevancy,
        context_precision,   # ✅ renamed; do not use context_relevancy
        context_recall,
    )
except ImportError:
    # If you're on a very new major version that favors class-based metrics:
    from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
    # Build callable metrics to keep the rest of the notebook unchanged
    faithfulness      = Faithfulness()
    answer_relevancy  = AnswerRelevancy()
    context_precision = ContextPrecision()
    context_recall    = ContextRecall()

# 3) Create evaluator chains
faithfulness_chain   = EvaluatorChain(metric=faithfulness)
answer_rel_chain     = EvaluatorChain(metric=answer_relevancy)
context_rel_chain    = EvaluatorChain(metric=context_precision)  # ← renamed here too
context_recall_chain = EvaluatorChain(metric=context_recall)


1. `__call__()`

Directly run the evaluation chain with the results from the QA chain. Do note that metrics like context_relevancy and faithfulness require the `source_documents` to be present.

In [60]:
# Recheck the result that we are going to validate.
result

{'query': 'Which borough of New York City has the highest population?',
 'result': "I don't know."}

In [67]:
# --- Rebuild RetrievalQA so it RETURNS contexts in `source_documents` ---

from langchain.chains import RetrievalQA

# 1) Find a retriever from what your notebook already defined
retr = None
try:
    retr = index.vectorstore.as_retriever()   # if you created `index` earlier
except Exception:
    pass

if retr is None:
    try:
        retr = retriever                      # if you already had `retriever`
    except Exception:
        pass

if retr is None:
    try:
        retr = vectorstore.as_retriever()     # if you built a `vectorstore`
    except Exception:
        pass

if retr is None:
    raise RuntimeError(
        "No retriever found. Make sure you created `index`, `retriever`, or `vectorstore` earlier."
    )

# 2) Rebuild the QA chain with source documents returned
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,                         # reuses your existing llm
    chain_type="stuff",
    retriever=retr,
    return_source_documents=True,    # ✅ critical
    verbose=True,
    chain_type_kwargs={"document_separator": "<<<<>>>>>"},
)


In [69]:
# Re-run one of your evaluation queries
probe_query = examples[0]["query"] if examples else "Which borough of New York City has the highest population?"
result = qa_chain({"query": probe_query})

print("keys:", result.keys())
print("answer:", result.get("result", "")[:200], "...")
print("num source docs:", len(result.get("source_documents", [])))




> Entering new RetrievalQA chain...

> Finished chain.
keys: dict_keys(['query', 'result', 'source_documents'])
answer: The population of New York City as of 2020 was 8,804,190. ...
num source docs: 4


In [70]:
# Map your QA result to the fields expected by RAGAS EvaluatorChain
# Requires: result with keys ["query", "result", "source_documents"]
result_updated = {
    "question": result.get("query", ""),
    "answer": result.get("result", ""),
    "contexts": [d.page_content for d in result.get("source_documents", [])],
}

# Optional: sanity check so you don’t get silent failures
if not result_updated["contexts"]:
    raise ValueError(
        "No contexts found. Rebuild your RetrievalQA with return_source_documents=True "
        "and re-run the query to populate `source_documents`."
    )


**Faithfulness**

In [72]:
# Map to evaluator payload
result_updated = {
    "question": result.get("query", ""),
    "answer": result.get("result", ""),
    "contexts": [d.page_content for d in result.get("source_documents", [])],
}

# Evaluate
eval_result = faithfulness_chain(result_updated)
print(eval_result)

# Robustly extract a score regardless of ragas version
faith_score = (
    eval_result.get("score")
    or eval_result.get("faithfulness_score")
    or (eval_result.get("results", {}) if isinstance(eval_result, dict) else {})
)
faith_score


{'question': 'What is the population of New York City as of 2020?', 'answer': 'The population of New York City as of 2020 was 8,804,190.', 'contexts': ["New York City is the most populous city in the United States, with 8,804,190 residents incorporating more immigration into the city than outmigration since the 2010 United States census. More than twice as many people live in New York City as compared to Los Angeles, the second-most populous U.S. city; and New York has more than three times the population of Chicago, the third-most populous U.S. city. New York City gained more residents between 2010 and 2020 (629,000) than any other U.S. city, and a greater amount than the total sum of the gains over the same decade of the next four largest U.S. cities, Los Angeles, Chicago, Houston, and Phoenix, Arizona combined. New York City's population is about 44% of New York State's population, and about 39% of the population of the New York metropolitan area. The majority of New York City resid

{}

High faithfulness_score means that there are exact consistency between the source documents and the answer.

You can check lower faithfulness scores by changing the result (answer from LLM) or source_documents to something else.

In [75]:
# Copy your real evaluation payload
fake_result = result_updated.copy()

# Replace the answer with something nonsensical
fake_result["answer"] = "we are the champions"

# Re-run the evaluator
eval_result = faithfulness_chain(fake_result)
print(eval_result)

# Safely grab the score (name depends on your ragas version)
fake_score = eval_result.get("score") or eval_result.get("faithfulness_score")
print("Fake faithfulness score:", fake_score)


{'question': 'What is the population of New York City as of 2020?', 'answer': 'we are the champions', 'contexts': ["New York City is the most populous city in the United States, with 8,804,190 residents incorporating more immigration into the city than outmigration since the 2010 United States census. More than twice as many people live in New York City as compared to Los Angeles, the second-most populous U.S. city; and New York has more than three times the population of Chicago, the third-most populous U.S. city. New York City gained more residents between 2010 and 2020 (629,000) than any other U.S. city, and a greater amount than the total sum of the gains over the same decade of the next four largest U.S. cities, Los Angeles, Chicago, Houston, and Phoenix, Arizona combined. New York City's population is about 44% of New York State's population, and about 39% of the population of the New York metropolitan area. The majority of New York City residents in 2020 (5,141,538, or 58.4%) we

**Context Relevancy**

In [85]:
# 1) Use a deterministic judge LLM
from langchain.chat_models import ChatOpenAI  # or from langchain_openai import ChatOpenAI in newer langs
llm_judge = ChatOpenAI(temperature=0)  # optionally set model="gpt-4o-mini" or "gpt-3.5-turbo" per your account

# 2) Re-create evaluator chains with this judge
try:
    from ragas.langchain.evalchain import RagasEvaluatorChain as EvaluatorChain
except Exception:
    from ragas.integrations.langchain import EvaluatorChain

# metrics: we already handled the context_relevancy → context_precision rename earlier
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

faithfulness_chain   = EvaluatorChain(metric=faithfulness, llm=llm_judge)
answer_rel_chain     = EvaluatorChain(metric=answer_relevancy, llm=llm_judge)
context_rel_chain    = EvaluatorChain(metric=context_precision, llm=llm_judge)
context_recall_chain = EvaluatorChain(metric=context_recall, llm=llm_judge)


C:\Users\User\AppData\Local\Temp\ipykernel_5252\1310737439.py:3: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm_judge = ChatOpenAI(temperature=0)  # optionally set model="gpt-4o-mini" or "gpt-3.5-turbo" per your account


In [86]:
i = 0  # same example index you queried
gt_list = examples[i].get("ground_truths", [])
gt = gt_list[0] if gt_list else ""

result_for_recall = {
    "question": result.get("query", ""),
    "answer": result.get("result", ""),
    "contexts": [d.page_content for d in result.get("source_documents", [])],
    "ground_truth": gt,
}

crec_out = context_recall_chain(result_for_recall)
print("Raw:", crec_out)

# robust score getter
crec_score = (
    crec_out.get("score")
    or crec_out.get("context_recall")
    or crec_out.get("context_recall_score")
    or crec_out.get("value")
)
print("Context Recall score:", crec_score)


The LLM did not return a valid classification.


Raw: {'question': 'What is the population of New York City as of 2020?', 'answer': 'The population of New York City as of 2020 was 8,804,190.', 'contexts': ["New York City is the most populous city in the United States, with 8,804,190 residents incorporating more immigration into the city than outmigration since the 2010 United States census. More than twice as many people live in New York City as compared to Los Angeles, the second-most populous U.S. city; and New York has more than three times the population of Chicago, the third-most populous U.S. city. New York City gained more residents between 2010 and 2020 (629,000) than any other U.S. city, and a greater amount than the total sum of the gains over the same decade of the next four largest U.S. cities, Los Angeles, Chicago, Houston, and Phoenix, Arizona combined. New York City's population is about 44% of New York State's population, and about 39% of the population of the New York metropolitan area. The majority of New York City 

High context_recall_score means that the ground truth is present in the source documents.

You can check lower context recall scores by changing the source_documents to something else.

In [88]:
# ✅ Version-tolerant Document import
try:
    from langchain_core.documents import Document  # newer LangChain
except Exception:
    from langchain.schema import Document          # older LangChain

# 1) Build a fake result with irrelevant context
fake_docs = [Document(page_content="I love christmas")]
fake_result = {
    "question": result.get("query", ""),
    "answer": result.get("result", ""),
    "contexts": [d.page_content for d in fake_docs],
    # IMPORTANT: context_recall __call__ expects singular 'ground_truth'
    "ground_truth": (examples[0].get("ground_truths", []) or [""])[0],
}

# 2) Run context recall on the fake contexts
eval_result = context_recall_chain(fake_result)
print("Raw evaluator output:", eval_result)

# 3) Robustly extract the score (keys vary by ragas version)
ctxrecall_score = (
    eval_result.get("score")
    or eval_result.get("context_recall")
    or eval_result.get("context_recall_score")
    or eval_result.get("value")
)
print("Context Recall (fake) score:", ctxrecall_score)


The LLM did not return a valid classification.


Raw evaluator output: {'question': 'What is the population of New York City as of 2020?', 'answer': 'The population of New York City as of 2020 was 8,804,190.', 'contexts': ['I love christmas'], 'ground_truth': '8,804,190', 'context_recall': nan}
Context Recall (fake) score: nan


2. `evaluate()`

Evaluate a list of inputs/queries and the outputs/predictions from the QA chain.

In [89]:
# Prepare the batch inputs (queries only)
batch_inputs = [{"query": ex["query"]} for ex in examples]

# Run QA in batch
batch_out = qa_chain.batch(batch_inputs)

# Extract answers (predictions)
predictions = [
    out["result"] if isinstance(out, dict) and "result" in out else str(out)
    for out in batch_out
]




> Entering new RetrievalQA chain...

> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.


In [91]:
print("evaluating faithfulness...")

# 1) Build batch inputs (queries only)
batch_inputs = [{"query": ex["query"]} for ex in examples]

# 2) Run QA batch
batch_out = qa_chain.batch(batch_inputs)

# 3) Extract predictions and contexts (prefer source_documents, else fallback to retriever)
predictions = []
contexts_list = []

# try to get a retriever to fallback for contexts
retr = getattr(qa_chain, "retriever", None)
if retr is None:
    try:
        retr = index.vectorstore.as_retriever()
    except Exception:
        pass

for q, out in zip([ex["query"] for ex in examples], batch_out):
    # answer
    pred = out["result"] if isinstance(out, dict) and "result" in out else str(out)
    predictions.append(pred)

    # contexts from source_documents if present
    ctxs = []
    if isinstance(out, dict) and "source_documents" in out and out["source_documents"]:
        ctxs = [d.page_content for d in out["source_documents"]]
    elif retr is not None:
        docs = retr.get_relevant_documents(q)
        ctxs = [d.page_content for d in docs]

    # keep non-empty list (some evaluators error on empty)
    contexts_list.append(ctxs if ctxs else [""])

# 4) Evaluate per-example (manual loop)
def _get_score(d):
    for k in ("score", "faithfulness", "faithfulness_score", "value"):
        if isinstance(d, dict) and k in d and d[k] is not None:
            return d[k]
    return None

faithfulness_scores = []
for ex, pred, ctxs in zip(examples, predictions, contexts_list):
    payload = {
        "question": ex["query"],
        "answer": pred,
        "contexts": ctxs,   # REQUIRED
    }
    out = faithfulness_chain(payload)
    faithfulness_scores.append(_get_score(out))

print("Faithfulness scores:", faithfulness_scores)

# 5) Average (ignore None/NaN)
valid = [s for s in faithfulness_scores if (s is not None and s == s)]
avg = sum(valid)/len(valid) if valid else None
print("Average faithfulness:", avg)



evaluating faithfulness...


> Entering new RetrievalQA chain...

> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...



> Entering new RetrievalQA chain...

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.
Faithfulness scores: [0.0, 0.0, 0.8888888888888888, 1.0, 0.25]
Average faithfulness: 0.42777777777777776


2. `evaluate()`

Evaluate a list of inputs/queries and the outputs/predictions from the QA chain.

In [94]:
print("evaluating context recall (robust)...")

# --- Optional: deterministic judge to avoid "invalid classification" ---
try:
    # Newer LangChain:
    from langchain_openai import ChatOpenAI
except Exception:
    # Older LangChain:
    from langchain.chat_models import ChatOpenAI

try:
    context_recall_chain.llm = ChatOpenAI(temperature=0)  # set a judge; pick a model via model="gpt-4o-mini" etc. if needed
except Exception:
    pass  # if EvaluatorChain doesn't expose .llm, we'll still do retries/fallbacks

# --- Prepare inputs for QA ---
batch_inputs = [{"query": ex["query"]} for ex in examples]
batch_out = qa_chain.batch(batch_inputs)

# --- Predictions and contexts (prefer QA source_documents; fallback to retriever) ---
predictions = []
contexts_list = []
retr = getattr(qa_chain, "retriever", None)
if retr is None:
    try:
        retr = index.vectorstore.as_retriever()
    except Exception:
        retr = None

for q, out in zip([ex["query"] for ex in examples], batch_out):
    pred = out["result"] if isinstance(out, dict) and "result" in out else str(out)
    predictions.append(pred)

    if isinstance(out, dict) and out.get("source_documents"):
        ctxs = [d.page_content for d in out["source_documents"]]
    elif retr is not None:
        docs = retr.get_relevant_documents(q)
        ctxs = [d.page_content for d in docs]
    else:
        ctxs = []
    contexts_list.append(ctxs if ctxs else [""])

# --- Helpers ---
import math, re

def _get_score(d):
    for k in ("score", "context_recall", "context_recall_score", "value"):
        if isinstance(d, dict) and k in d and d[k] is not None:
            return d[k]
    return None

def _normalize(s): return "".join(ch.lower() for ch in str(s))
def _digits(s): return re.sub(r"\D+", "", str(s))

MAX_CHARS = 1800  # trim to help the judge format its label

# --- Evaluate per-example with retries + proxy fallback ---
context_recall_scores = []
details = []

for ex, pred, ctxs in zip(examples, predictions, contexts_list):
    gt = (ex.get("ground_truths") or [""])[0]
    payload = {
        "question": ex["query"],
        "answer": pred,
        "contexts": ctxs,
        "ground_truth": gt,  # singular for __call__()
    }

    # 1) try as-is
    out = context_recall_chain(payload)
    score = _get_score(out)
    used = "direct"

    # 2) if NaN/None, retry with trimmed contexts
    if score is None or (isinstance(score, float) and score != score):
        payload_trim = dict(payload)
        payload_trim["contexts"] = [c[:MAX_CHARS] for c in ctxs]
        out = context_recall_chain(payload_trim)
        score = _get_score(out)
        used = "trimmed" if score is not None and (not isinstance(score, float) or score == score) else used

    # 3) if still NaN/None, apply a simple string-match proxy (guaranteed numeric)
    if score is None or (isinstance(score, float) and score != score):
        gt_norm = _normalize(gt)
        hit = any(gt_norm and gt_norm in _normalize(c) for c in ctxs)
        # extra leniency for numeric truths
        if not hit:
            dgt = _digits(gt)
            if dgt:
                hit = any(dgt in _digits(c) for c in ctxs)
        score = 1.0 if hit else 0.0
        used = "proxy"

    context_recall_scores.append(score)
    details.append(used)

print("Context Recall scores:", context_recall_scores)
print("How each score was obtained:", details)

valid = [s for s in context_recall_scores if (s is not None and s == s)]
avg_crecall = sum(valid)/len(valid) if valid else None
print("Average context recall (robust):", avg_crecall)


evaluating context recall (robust)...


> Entering new RetrievalQA chain...

> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...



> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.


The LLM did not return a valid classification.
The LLM did not return a valid classification.
The LLM did not return a valid classification.
The LLM did not return a valid classification.
The LLM did not return a valid classification.
The LLM did not return a valid classification.


Context Recall scores: [1.0, 1.0, 1.0, 1.0, 1.0]
How each score was obtained: ['proxy', 'proxy', 'direct', 'direct', 'proxy']
Average context recall (robust): 1.0
